# STRAT-002 v5 - Production Backtest

**Strategy:** SOPR + Realized Loss Entry + Simple 30% Trail Exit

**Entry:**
- SOPR < 1
- STH-SOPR < 1  
- Realized Loss Z-Score > 0.5

**Exit:**
- 30% trailing stop from peak (always active)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import vectorbt as vbt
from numba import njit
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print(f"VectorBT version: {vbt.__version__}")
print("STRAT-002 v5 Production Backtest 📊")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()

# Create RL z-score
df['rl_ma30'] = df['realized_loss'].rolling(30).mean()
df['rl_std30'] = df['realized_loss'].rolling(30).std()
df['rl_zscore'] = (df['realized_loss'] - df['rl_ma30']) / df['rl_std30']

# Filter to test period
df = df[df.index >= '2018-12-15'].dropna()

print(f"Data: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")

---
## 1. Define Entry Signals

In [ ]:
# Strategy parameters
RL_Z_THRESHOLD = 0.5
TRAIL_PCT = 0.30  # 30% trailing stop

# Entry signal: SOPR < 1 AND STH-SOPR < 1 AND RL Z > 0.5
entry_condition = (
    (df['sopr'] < 1) & 
    (df['sopr_sth'] < 1) & 
    (df['rl_zscore'] > RL_Z_THRESHOLD)
)

# First day of signal only
entries = entry_condition & ~entry_condition.shift(1).fillna(False)

print(f"Entry signals: {entries.sum()}")
print(f"\nEntry dates:")
for date in entries[entries].index:
    print(f"  {date.date()}: ${df.loc[date, 'price']:,.0f}")

---
## 2. Custom Backtest with Simple Trailing Stop

In [ ]:
@njit
def simple_trail_exit(price_arr, entry_idx, trail_pct=0.30):
    """
    Simple trailing stop - always active from entry.
    Exit when price drops trail_pct from peak.
    """
    entry_price = price_arr[entry_idx]
    peak_price = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        current_price = price_arr[j]
        
        # Update peak
        if current_price > peak_price:
            peak_price = current_price
        
        # Check trail stop
        trail_stop = peak_price * (1 - trail_pct)
        if current_price <= trail_stop:
            return j, trail_stop
    
    # End of data
    return len(price_arr) - 1, price_arr[-1]

In [ ]:
def run_backtest(df, entries, trail_pct=0.30, initial_capital=100000, fees=0.001):
    """
    Run backtest with simple trailing stop.
    """
    price_arr = df['price'].values
    dates = df.index
    
    entry_indices = np.where(entries.values)[0]
    
    trades = []
    equity_curve = [initial_capital]
    equity_dates = [dates[0]]
    
    i = 0
    current_equity = initial_capital
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        entry_date = dates[entry_idx]
        entry_price = price_arr[entry_idx]
        
        # Get exit
        exit_idx, exit_price = simple_trail_exit(price_arr, entry_idx, trail_pct)
        exit_date = dates[exit_idx]
        
        # Calculate returns
        gross_return = (exit_price / entry_price) - 1
        net_return = gross_return - (2 * fees)  # Entry + exit fees
        
        # Track peak during trade
        trade_prices = price_arr[entry_idx:exit_idx+1]
        peak_price = trade_prices.max()
        max_gain = (peak_price / entry_price) - 1
        
        # Update equity
        current_equity = current_equity * (1 + net_return)
        
        trades.append({
            'entry_date': entry_date,
            'exit_date': exit_date,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'peak_price': peak_price,
            'gross_return': gross_return,
            'net_return': net_return,
            'max_gain': max_gain,
            'days_held': (exit_date - entry_date).days,
            'equity_after': current_equity
        })
        
        equity_curve.append(current_equity)
        equity_dates.append(exit_date)
        
        # Skip entries during this trade
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    equity_df = pd.DataFrame({'equity': equity_curve}, index=equity_dates)
    
    return trades_df, equity_df

In [ ]:
# Run backtest
INITIAL_CAPITAL = 100000
FEES = 0.001  # 0.1% per trade

trades, equity = run_backtest(df, entries, TRAIL_PCT, INITIAL_CAPITAL, FEES)

print("TRADE LOG")
print("="*130)
print(f"{'Entry':<12} {'Exit':<12} {'Entry $':>10} {'Peak $':>10} {'Exit $':>10} {'Max Gain':>10} {'Net':>10} {'Days':>6} {'Equity':>14}")
print("-"*130)

for _, t in trades.iterrows():
    print(f"{str(t['entry_date'].date()):<12} {str(t['exit_date'].date()):<12} "
          f"{t['entry_price']:>10,.0f} {t['peak_price']:>10,.0f} {t['exit_price']:>10,.0f} "
          f"{t['max_gain']*100:>+9.0f}% {t['net_return']*100:>+9.0f}% "
          f"{t['days_held']:>6} ${t['equity_after']:>13,.0f}")

---
## 3. Performance Metrics

In [ ]:
def calculate_metrics(trades, equity, initial_capital, df):
    """Calculate comprehensive performance metrics."""
    
    if len(trades) == 0:
        return {}
    
    # Basic stats
    total_trades = len(trades)
    winners = trades[trades['net_return'] > 0]
    losers = trades[trades['net_return'] <= 0]
    
    win_rate = len(winners) / total_trades
    
    # Returns
    final_equity = trades['equity_after'].iloc[-1]
    total_return = (final_equity / initial_capital) - 1
    
    # CAGR
    start_date = trades['entry_date'].iloc[0]
    end_date = trades['exit_date'].iloc[-1]
    years = (end_date - start_date).days / 365.25
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    
    # Average trade
    avg_return = trades['net_return'].mean()
    avg_winner = winners['net_return'].mean() if len(winners) > 0 else 0
    avg_loser = losers['net_return'].mean() if len(losers) > 0 else 0
    
    # Profit factor
    gross_profit = winners['net_return'].sum() if len(winners) > 0 else 0
    gross_loss = abs(losers['net_return'].sum()) if len(losers) > 0 else 0.0001
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    
    # Max drawdown
    equity_values = equity['equity'].values
    peak = equity_values[0]
    max_dd = 0
    for eq in equity_values:
        if eq > peak:
            peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd:
            max_dd = dd
    
    # Sharpe (annualized)
    returns = trades['net_return'].values
    trades_per_year = total_trades / years
    if len(returns) > 1 and returns.std() > 0:
        sharpe = (returns.mean() / returns.std()) * np.sqrt(trades_per_year)
    else:
        sharpe = 0
    
    # Sortino
    downside_returns = returns[returns < 0]
    if len(downside_returns) > 0 and np.std(downside_returns) > 0:
        sortino = (returns.mean() / np.std(downside_returns)) * np.sqrt(trades_per_year)
    else:
        sortino = float('inf')
    
    # Buy & hold comparison
    bh_start_price = df.loc[trades['entry_date'].iloc[0], 'price']
    bh_end_price = df.loc[trades['exit_date'].iloc[-1], 'price']
    bh_return = (bh_end_price / bh_start_price) - 1
    bh_cagr = (1 + bh_return) ** (1 / years) - 1 if years > 0 else 0
    
    # Average days
    avg_days = trades['days_held'].mean()
    
    return {
        'total_trades': total_trades,
        'winners': len(winners),
        'losers': len(losers),
        'win_rate': win_rate,
        'total_return': total_return,
        'cagr': cagr,
        'avg_return': avg_return,
        'avg_winner': avg_winner,
        'avg_loser': avg_loser,
        'profit_factor': profit_factor,
        'max_drawdown': max_dd,
        'sharpe': sharpe,
        'sortino': sortino,
        'avg_days_held': avg_days,
        'bh_return': bh_return,
        'bh_cagr': bh_cagr,
        'alpha': cagr - bh_cagr,
        'start_date': start_date,
        'end_date': end_date,
        'years': years,
        'final_equity': final_equity
    }

In [ ]:
metrics = calculate_metrics(trades, equity, INITIAL_CAPITAL, df)

print("\n" + "="*80)
print("PERFORMANCE REPORT: STRAT-002 v5")
print("="*80)

print(f"\n📅 PERIOD")
print(f"   Start: {metrics['start_date'].date()}")
print(f"   End: {metrics['end_date'].date()}")
print(f"   Duration: {metrics['years']:.1f} years")

print(f"\n💰 RETURNS")
print(f"   Total Return: {metrics['total_return']*100:+,.0f}%")
print(f"   CAGR: {metrics['cagr']*100:+.1f}%")
print(f"   Buy & Hold Return: {metrics['bh_return']*100:+,.0f}%")
print(f"   Buy & Hold CAGR: {metrics['bh_cagr']*100:+.1f}%")
print(f"   Alpha (CAGR - BH): {metrics['alpha']*100:+.1f}%")

print(f"\n📊 TRADE STATISTICS")
print(f"   Total Trades: {metrics['total_trades']}")
print(f"   Winners: {metrics['winners']} ({metrics['win_rate']*100:.0f}%)")
print(f"   Losers: {metrics['losers']} ({(1-metrics['win_rate'])*100:.0f}%)")
print(f"   Avg Trade: {metrics['avg_return']*100:+.1f}%")
print(f"   Avg Winner: {metrics['avg_winner']*100:+.1f}%")
print(f"   Avg Loser: {metrics['avg_loser']*100:.1f}%")
print(f"   Avg Days Held: {metrics['avg_days_held']:.0f}")

print(f"\n📈 RISK METRICS")
print(f"   Profit Factor: {metrics['profit_factor']:.2f}")
print(f"   Max Drawdown: {metrics['max_drawdown']*100:.1f}%")
print(f"   Sharpe Ratio: {metrics['sharpe']:.2f}")
print(f"   Sortino Ratio: {metrics['sortino']:.2f}")

print(f"\n💵 CAPITAL")
print(f"   Initial: ${INITIAL_CAPITAL:,}")
print(f"   Final: ${metrics['final_equity']:,.0f}")

print("\n" + "="*80)

---
## 4. VectorBT Portfolio Comparison

In [ ]:
# Use VectorBT's built-in trailing stop for comparison
close = df['price']

# Create exit signals using trailing stop
# VectorBT has built-in trailing stop functionality
pf = vbt.Portfolio.from_signals(
    close,
    entries=entries,
    exits=None,  # Use stop loss instead
    sl_stop=None,
    sl_trail=True,  # Trailing stop
    tp_stop=None,
    stop_exit_price='close',
    fees=FEES,
    init_cash=INITIAL_CAPITAL,
    freq='D'
)

print("\nVectorBT Portfolio Stats (for reference):")
print("="*60)
print(pf.stats())

---
## 5. Visualization

In [ ]:
# Create comprehensive chart
fig = make_subplots(
    rows=4, cols=1, 
    shared_xaxes=True, 
    row_heights=[0.4, 0.2, 0.2, 0.2],
    subplot_titles=['Equity Curve vs Buy & Hold', 'BTC Price with Trades', 'Drawdown', 'Trade Returns']
)

# 1. Equity curve
fig.add_trace(go.Scatter(
    x=equity.index, y=equity['equity'],
    name='Strategy', line=dict(color='green', width=2)
), row=1, col=1)

# Buy & hold equity
bh_start_price = df.loc[trades['entry_date'].iloc[0], 'price']
bh_equity = INITIAL_CAPITAL * (df['price'] / bh_start_price)
fig.add_trace(go.Scatter(
    x=df.index, y=bh_equity,
    name='Buy & Hold', line=dict(color='gray', width=1, dash='dot')
), row=1, col=1)

# 2. Price with trades
fig.add_trace(go.Scatter(
    x=df.index, y=df['price'],
    name='BTC Price', line=dict(color='orange', width=1)
), row=2, col=1)

# Entry markers
fig.add_trace(go.Scatter(
    x=trades['entry_date'], y=trades['entry_price'],
    mode='markers', name='Entry',
    marker=dict(color='green', size=10, symbol='triangle-up')
), row=2, col=1)

# Exit markers
fig.add_trace(go.Scatter(
    x=trades['exit_date'], y=trades['exit_price'],
    mode='markers', name='Exit',
    marker=dict(color='red', size=10, symbol='triangle-down')
), row=2, col=1)

# 3. Drawdown
equity_series = equity['equity']
rolling_max = equity_series.expanding().max()
drawdown = (equity_series - rolling_max) / rolling_max * 100

fig.add_trace(go.Scatter(
    x=drawdown.index, y=drawdown,
    fill='tozeroy', name='Drawdown %',
    line=dict(color='red')
), row=3, col=1)

# 4. Trade returns
colors = ['green' if r > 0 else 'red' for r in trades['net_return']]
fig.add_trace(go.Bar(
    x=trades['exit_date'], y=trades['net_return'] * 100,
    name='Trade Return %',
    marker_color=colors
), row=4, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_yaxes(type='log', row=2, col=1)
fig.update_layout(height=1000, title_text='STRAT-002 v5 Backtest Results')
fig.show()

In [ ]:
# Trade return distribution
fig = make_subplots(rows=1, cols=2, subplot_titles=['Return Distribution', 'Return by Trade'])

# Histogram
fig.add_trace(go.Histogram(
    x=trades['net_return'] * 100,
    nbinsx=15,
    name='Returns',
    marker_color='blue'
), row=1, col=1)

fig.add_vline(x=0, line_dash='dash', line_color='black', row=1, col=1)
fig.add_vline(x=trades['net_return'].mean()*100, line_dash='dash', line_color='green', row=1, col=1)

# Scatter by trade
fig.add_trace(go.Scatter(
    x=list(range(1, len(trades)+1)),
    y=trades['net_return'] * 100,
    mode='markers+lines',
    name='Trade Returns',
    marker=dict(color=['green' if r > 0 else 'red' for r in trades['net_return']], size=10)
), row=1, col=2)

fig.add_hline(y=0, line_dash='dash', line_color='black', row=1, col=2)

fig.update_layout(height=400, title_text='Trade Analysis')
fig.show()

---
## 6. Final Summary

In [ ]:
print("\n" + "="*80)
print("STRAT-002 v5 FINAL SUMMARY")
print("="*80)

print(f"""
STRATEGY: SOPR + Realized Loss + Simple 30% Trail

ENTRY RULES:
  - SOPR < 1 (market selling at loss)
  - AND STH-SOPR < 1 (short-term holders selling at loss)
  - AND Realized Loss Z-Score > {RL_Z_THRESHOLD} (elevated losses)

EXIT RULES:
  - {TRAIL_PCT*100:.0f}% trailing stop from peak (always active)
  - That's it. Simple.

PERFORMANCE ({metrics['start_date'].date()} to {metrics['end_date'].date()}):
  
  Strategy Return:    {metrics['total_return']*100:+,.0f}%
  Buy & Hold Return:  {metrics['bh_return']*100:+,.0f}%
  
  Strategy CAGR:      {metrics['cagr']*100:+.1f}%
  Buy & Hold CAGR:    {metrics['bh_cagr']*100:+.1f}%
  Alpha:              {metrics['alpha']*100:+.1f}%
  
  Win Rate:           {metrics['win_rate']*100:.0f}%
  Profit Factor:      {metrics['profit_factor']:.2f}
  Sharpe Ratio:       {metrics['sharpe']:.2f}
  Sortino Ratio:      {metrics['sortino']:.2f}
  Max Drawdown:       {metrics['max_drawdown']*100:.1f}%
  
  ${INITIAL_CAPITAL:,} → ${metrics['final_equity']:,.0f}
""")

print("="*80)

In [ ]:
# Save results
import json

trades_export = trades.copy()
trades_export['entry_date'] = trades_export['entry_date'].astype(str)
trades_export['exit_date'] = trades_export['exit_date'].astype(str)

results = {
    'strategy': 'STRAT-002-v5',
    'parameters': {
        'rl_z_threshold': RL_Z_THRESHOLD,
        'trail_pct': TRAIL_PCT,
        'initial_capital': INITIAL_CAPITAL,
        'fees': FEES
    },
    'metrics': {
        'total_return': float(metrics['total_return']),
        'cagr': float(metrics['cagr']),
        'win_rate': float(metrics['win_rate']),
        'profit_factor': float(metrics['profit_factor']),
        'sharpe': float(metrics['sharpe']),
        'sortino': float(metrics['sortino']),
        'max_drawdown': float(metrics['max_drawdown']),
        'total_trades': int(metrics['total_trades']),
        'avg_days_held': float(metrics['avg_days_held']),
        'bh_return': float(metrics['bh_return']),
        'alpha': float(metrics['alpha']),
        'final_equity': float(metrics['final_equity']),
        'start_date': str(metrics['start_date'].date()),
        'end_date': str(metrics['end_date'].date()),
        'years': float(metrics['years'])
    },
    'trades': trades_export.to_dict('records')
}

with open('../data/strat002_v5_backtest_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to ../data/strat002_v5_backtest_results.json")